# Análise Exploratória -- Marketing Attribuition Model Comparison

# Objective

Identify which channels contribute the most to conversions and how combining them can maximize results.

# Main Question

which channels contribute the most to customer conversions over time, and how can they be combined to maximize performance?

# Secondary Questions
- which channels contribute the most to conversions?
- which channels perform better together?
- Are there underestimated channels?
- How do different attribution models change this perspective?


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../data/raw/data.csv", encoding="ISO-8859-1")

In [3]:
df.head()

,User ID,Timestamp,Channel,Campaign,Conversion
0,83281,2025-02-10 07:58:51,Email,New Product Launch,No
1,68071,2025-02-10 23:38:48,Search Ads,Winter Sale,No
2,90131,2025-02-11 10:41:07,Social Media,Brand Awareness,Yes
3,71026,2025-02-10 08:19:44,Direct Traffic,-,Yes
4,94486,2025-02-10 15:15:46,Email,Retargeting,Yes


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   User ID     10000 non-null  int64
 1   Timestamp   10000 non-null  str  
 2   Channel     10000 non-null  str  
 3   Campaign    10000 non-null  str  
 4   Conversion  10000 non-null  str  
dtypes: int64(1), str(4)
memory usage: 390.8 KB


In [5]:
df.describe()

,User ID
count,10000.00000
mean,54957.01700
std,25685.14741
min,10028.00000
25%,32391.00000
50%,55446.00000
75%,77298.00000
max,99995.00000


# Analyzing which channels contribute the most to conversion

In [6]:
#Conversão por canal
df["Conversion"] = df["Conversion"].map({"Yes": 1, "No": 0})
df.groupby("Channel")["Conversion"].sum().sort_values(ascending=False)

Channel
Direct Traffic    853
Referral          841
Email             830
Display Ads       828
Social Media      820
Search Ads        772
Name: Conversion, dtype: int64

In [7]:
# Taxa de conversão por canal
df.groupby("Channel")["Conversion"].mean().sort_values(ascending= False)

Channel
Email             0.501814
Referral          0.499110
Display Ads       0.496105
Direct Traffic    0.495642
Social Media      0.493381
Search Ads        0.479801
Name: Conversion, dtype: float64

In [8]:
# Volume X Conversão
df.groupby("Channel")["Conversion"].agg(["count", "mean", "sum"])

,count,mean,sum
Channel,,,
Direct Traffic,1721,0.495642,853
Display Ads,1669,0.496105,828
Email,1654,0.501814,830
Referral,1685,0.499110,841
Search Ads,1609,0.479801,772
Social Media,1662,0.493381,820


The difference between conversion volume and conversion rate suggests that different channels play distinct roles throughout the customer journey. While Direct Traffic predominantly acts at the final stage of conversion, Email stands out for its efficiency, possibly associated with remarketing strategies and engagement with already qualified users.

# Analyzing which channels perform better together

Creating user journeys to identify which channels contributed to conversions.

In [9]:
df_sorted = df.sort_values(by=["User ID", "Timestamp"])

journeys = df_sorted.groupby("User ID")["Channel"].apply(list)

journeys_df = journeys.reset_index()

print(journeys_df.head(10))

   User ID                                            Channel
0    10028                          [Search Ads, Display Ads]
1    10045                          [Search Ads, Display Ads]
2    10062              [Social Media, Direct Traffic, Email]
3    10068  [Search Ads, Social Media, Social Media, Searc...
4    10095  [Display Ads, Email, Referral, Display Ads, Se...
5    10120  [Referral, Email, Referral, Direct Traffic, Di...
6    10167           [Social Media, Display Ads, Display Ads]
7    10172                                  [Email, Referral]
8    10202  [Display Ads, Direct Traffic, Display Ads, Soc...
9    10255  [Social Media, Referral, Direct Traffic, Socia...


In [10]:
converted_users = df.groupby("User ID")["Conversion"].max()

converted_journeys = journeys_df[converted_users.values == 1]

converted_journeys.head()

,User ID,Channel
0,10028,"[Search Ads, Display Ads]"
1,10045,"[Search Ads, Display Ads]"
2,10062,"[Social Media, Direct Traffic, Email]"
3,10068,"[Search Ads, Social Media, Social Media, Searc..."
4,10095,"[Display Ads, Email, Referral, Display Ads, Se..."


# Evaluating which channels work best together.

In [11]:
converted_users= df.groupby("User ID")["Conversion"].max().reset_index()

journeys_df = journeys_df.merge(converted_users, on="User ID")

converted_journeys = journeys_df[journeys_df["Conversion"]==1]

converted_journeys.head()

,User ID,Channel,Conversion
0,10028,"[Search Ads, Display Ads]",1
1,10045,"[Search Ads, Display Ads]",1
2,10062,"[Social Media, Direct Traffic, Email]",1
3,10068,"[Search Ads, Social Media, Social Media, Searc...",1
4,10095,"[Display Ads, Email, Referral, Display Ads, Se...",1


In [12]:
from itertools import combinations
from collections import Counter

pair_counts = Counter()

for journey in converted_journeys["Channel"]:
    unique_channels = set(journey)
    pairs = combinations(unique_channels, 2)
    pair_counts.update(pairs)

pair_counts.most_common(10)

[(('Direct Traffic', 'Display Ads'), 523),
 (('Email', 'Search Ads'), 522),
 (('Social Media', 'Direct Traffic'), 520),
 (('Social Media', 'Search Ads'), 511),
 (('Email', 'Direct Traffic'), 504),
 (('Email', 'Display Ads'), 503),
 (('Social Media', 'Display Ads'), 503),
 (('Direct Traffic', 'Search Ads'), 497),
 (('Referral', 'Direct Traffic'), 470),
 (('Referral', 'Display Ads'), 452)]

# Creating the Final_df

In [13]:

user_channels = df.groupby("User ID")["Channel"].apply(set).reset_index()


channels_df = user_channels["Channel"].apply(
    lambda x: pd.Series({ch: 1 for ch in x})
).fillna(0)


conversion = df.groupby("User ID")["Conversion"].max().reset_index()

final_df = pd.concat(
    [user_channels["User ID"], channels_df, conversion["Conversion"]],
    axis=1
)

final_df.head()

,User ID,Search Ads,Display Ads,Social Media,Email,Direct Traffic,Referral,Conversion
0,10028,1.0,1.0,0.0,0.0,0.0,0.0,1
1,10045,1.0,1.0,0.0,0.0,0.0,0.0,1
2,10062,0.0,0.0,1.0,1.0,1.0,0.0,1
3,10068,1.0,0.0,1.0,0.0,0.0,0.0,1
4,10095,1.0,1.0,0.0,1.0,0.0,1.0,1


Analyzing which channels truly achieve higher conversion rates when combined.


In [14]:
from itertools import combinations
import pandas as pd

results = []

channels = [col for col in final_df.columns if col not in ["User ID", "Conversion"]]

for ch1, ch2 in combinations(channels, 2):
    both= final_df[(final_df[ch1] ==1) & (final_df[ch2]==1)]
    only_ch1 = final_df[(final_df[ch1] ==1) & (final_df[ch2] ==0)]
    only_ch2 = final_df[(final_df[ch1] ==0) & (final_df[ch2] ==1)]

    results.append({
        "Channel A": ch1,
        "Channel B": ch2,
        "Conv Both": both["Conversion"].mean(),
        "Conv Only A": only_ch1["Conversion"].mean(),
        "Conv Only B": only_ch2["Conversion"].mean(),
        "N Both": len(both),
        "N Only A": len(only_ch1),
        "N Only B": len(only_ch2)
    })

results_df = pd.DataFrame(results)

results_df.sort_values(by="Conv Both", ascending=False).head(10)


,Channel A,Channel B,Conv Both,Conv Only A,Conv Only B,N Both,N Only A,N Only B
0,Search Ads,Display Ads,0.957935,0.817935,0.871409,523,736,731
8,Display Ads,Referral,0.954710,0.870370,0.841678,552,702,739
6,Display Ads,Email,0.949057,0.877072,0.840599,530,724,734
7,Display Ads,Direct Traffic,0.945750,0.877318,0.840599,553,701,734
14,Direct Traffic,Referral,0.937500,0.843882,0.851748,576,711,715
2,Search Ads,Email,0.937163,0.827635,0.845827,557,702,707
5,Display Ads,Social Media,0.936685,0.885635,0.850267,537,717,748
11,Social Media,Referral,0.935540,0.846695,0.853556,574,711,717
12,Email,Direct Traffic,0.935065,0.849655,0.850267,539,725,748
1,Search Ads,Social Media,0.934186,0.831461,0.850949,547,712,738


The analysis indicates that channel combinations outperform isolated channel usage, especially combinations involving Display Ads and Search Ads, which demonstrate strong synergy with other touchpoints. However, the results also suggest that the increase in conversions is more strongly related to multi-channel exposure than to specific isolated combinations.

# Analysing Underestimated Channels

In [15]:
from collections import Counter

channel_presence = Counter()

for journey in converted_journeys["Channel"]:
    channel_presence.update(set(journey))

presence_df = pd.DataFrame(
    channel_presence.items(),
    columns=["Channel", "Presence in Converted Journeys"]
)

presence_df.sort_values(
    by="Presence in Converted Journeys",
    ascending=False
)

,Channel,Presence in Converted Journeys
5,Referral,1149
4,Direct Traffic,1140
2,Social Media,1139
1,Display Ads,1138
3,Email,1120
0,Search Ads,1103


The analysis indicates that channels traditionally associated with the top of the funnel, such as Social Media and Display Ads, may be partially underestimated by direct conversion metrics. Although they do not lead final conversions, these channels show a strong presence in converted journeys, suggesting a significant contribution to building purchase intent.

# Analysing How Different Attribuition Models Completely Change Channel Interpretation

In [16]:
last_click = []

for journeys in converted_journeys["Channel"]:
    last_click.append(journey[-1])

last_click_df = pd.Series(last_click).value_counts()

print(last_click_df)


Search Ads    2381
Name: count, dtype: int64


In [17]:
first_click = []

for journey in converted_journeys["Channel"]:
    first_click.append(journey[0])

first_click_df = pd.Series(first_click).value_counts()
print(first_click_df)

Display Ads       428
Direct Traffic    411
Referral          408
Social Media      389
Email             374
Search Ads        371
Name: count, dtype: int64


In [18]:
from collections import defaultdict

linear_scores = defaultdict(float)

for journey in converted_journeys["Channel"]:
    
    unique_channels = list(set(journey))
    
    credit = 1 / len(unique_channels)
    
    for channel in unique_channels:
        linear_scores[channel] += credit

linear_df = pd.Series(linear_scores).sort_values(ascending=False)

print(linear_df)

Display Ads       404.800000
Direct Traffic    402.516667
Referral          401.116667
Social Media      398.800000
Email             388.483333
Search Ads        385.283333
dtype: float64


In [19]:
comparison_df = pd.DataFrame({
    "Last Click": last_click_df,
    "First Click": first_click_df,
    "Linear": linear_df
}).fillna(0)

comparison_df

,Last Click,First Click,Linear
Direct Traffic,0.0,411,402.516667
Display Ads,0.0,428,404.800000
Email,0.0,374,388.483333
Referral,0.0,408,401.116667
Search Ads,2381.0,371,385.283333
Social Media,0.0,389,398.800000


# Final Conclusion


The analysis demonstrated that different attribution models produce significantly different interpretations of channel impact. While approaches focused on the last touchpoint favor channels closer to conversion, multi-touch models highlight the importance of awareness channels and the combined contribution of multiple touchpoints throughout the customer journey.

# Final Recommendation


It is recommended to adopt multi-channel strategies and more comprehensive attribuition models to better understand the real impact of marketing channels on convertions.